## Exercise

Build a prediction script on the first 64 examples of `wikiann` (`en`) using:
- BERT NER model: `dslim/bert-base-NER`
- Llama 1B chat model (for entity extraction via prompt)

The output must be a CSV file with columns:
- `id`
- `bert_prediction`
- `llama_prediction`

Below you have a draft solution scaffold: some pieces are complete, while key parts are left as TODOs for you to implement.

In [1]:
import os, json
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

print('Transformers version:', __import__('transformers').__version__)
print('Torch version:', torch.__version__)

BASE_DIR = os.path.join('lab2')
DATA_DIR = os.path.join(BASE_DIR, 'data')
CACHE_DIR = os.path.join(BASE_DIR, 'models_cache')
os.makedirs(CACHE_DIR, exist_ok=True)


Transformers version: 5.0.0
Torch version: 2.10.0+cu128


### Exercise: Fine-tune BERT on WikiANN

> Goal: train the token-classification model on the WikiANN train split and evaluate on validation.

Fill in the blanks in the next cell. Focus on data preprocessing, label alignment, and training args.

In [2]:
# Exercise scaffold: fill in all TODOs.
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
 )

# -----------------------------
# 1) Load WikiANN
# -----------------------------
wikiann = load_dataset('wikiann', 'en')
N_TRAIN_EXAMPLES = 10000 # TODO: select a subset of the training set for faster experimentation
wikiann['train'] = wikiann['train'].select(range(N_TRAIN_EXAMPLES))
label_list = wikiann['train'].features['ner_tags'].feature.names
label_to_id = {label: i for i, label in enumerate(label_list)}
id_to_label = {i: label for label, i in label_to_id.items()}

# -----------------------------
# 2) Tokenizer + label alignment
# -----------------------------
model_checkpoint = 'dslim/bert-base-NER'
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples['tokens'],
        is_split_into_words=True,
        truncation=True,
    )
    # TODO: Align the word-level labels to token-level labels.
    # - Use word_ids() to map tokens back to word indices.
    # - Use -100 for special tokens so they are ignored in loss.
    # - For subword tokens, keep the same label or convert I- tags as needed.
    aligned_labels = []
    # TODO: implement label alignment
    for i, label_ids in enumerate(examples['ner_tags']):
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids_aligned = []
        for word_idx in word_ids:
            if word_idx is None:
                # Special tokens ([CLS], [SEP], padding) → ignored in loss
                label_ids_aligned.append(-100)
            elif word_idx != previous_word_idx:
                # First token of a new word → assign the actual label
                label_ids_aligned.append(label_ids[word_idx])
            else:
                # Subsequent subword tokens of the same word
                # Keep the same label but convert B- to I- to stay BIO-consistent
                current_label = label_ids[word_idx]
                label_name = label_list[current_label]
                if label_name.startswith('B-'):
                    # Convert B- to the corresponding I- tag
                    i_label_name = 'I-' + label_name[2:]
                    label_ids_aligned.append(label_to_id.get(i_label_name, current_label))
                else:
                    label_ids_aligned.append(current_label)
            previous_word_idx = word_idx
        aligned_labels.append(label_ids_aligned)

    tokenized['labels'] = aligned_labels
    return tokenized

tokenized = wikiann.map(tokenize_and_align_labels, batched=True)

# -----------------------------
# 3) Model
# -----------------------------
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id_to_label,
    label2id=label_to_id,
    ignore_mismatched_sizes=True
 )

# -----------------------------
# 4) Training setup
# -----------------------------
training_args = TrainingArguments(
    output_dir='lab4/models_cache/wikiann_bert_ner',
    eval_strategy='epoch',        
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    data_collator=data_collator,
 )

# -----------------------------
# 5) Train + evaluate
# -----------------------------
# TODO: run training and evaluation
trainer.train()
metrics = trainer.evaluate()
print(metrics)

README.md: 0.00B [00:00, ?B/s]

en/validation-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/test-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/train-00000-of-00001.parquet:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.6566921472549438, 'eval_runtime': 19.6488, 'eval_samples_per_second': 508.936, 'eval_steps_per_second': 7.99, 'epoch': 5.0}


In [7]:
from seqeval.metrics import f1_score, classification_report
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

for ckpt in ['checkpoint-500', 'checkpoint-1000', 'checkpoint-1500', 'checkpoint-1565']:
    path = f'lab4/models_cache/wikiann_bert_ner/{ckpt}'
    tok = AutoTokenizer.from_pretrained(path)
    mdl = AutoModelForTokenClassification.from_pretrained(path).to(device)
    mdl.eval()
    
    preds, labs, _ = trainer.predict(tokenized['validation'])
    preds = preds.argmax(axis=-1)
    
    true_labels, true_preds = [], []
    for pred_seq, label_seq in zip(preds, labs):
        tl, tp = [], []
        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                tl.append(id_to_label[l])
                tp.append(mdl.config.id2label[p])
        true_labels.append(tl)
        true_preds.append(tp)
    
    print(f'{ckpt} → F1: {f1_score(true_labels, true_preds):.4f}')
    del mdl
    torch.cuda.empty_cache()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

checkpoint-500 → F1: 0.7963


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

checkpoint-1000 → F1: 0.7963


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

checkpoint-1500 → F1: 0.7963


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

checkpoint-1565 → F1: 0.7963


In [6]:
!pip install seqeval -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [8]:
from seqeval.metrics import f1_score, classification_report

# Get predictions on validation set
predictions, labels, _ = trainer.predict(tokenized['validation'])
predictions = predictions.argmax(axis=-1)

# Remove ignored index (-100)
true_labels = []
true_preds = []

for pred_seq, label_seq in zip(predictions, labels):
    true_label_seq = []
    true_pred_seq = []
    for p, l in zip(pred_seq, label_seq):
        if l != -100:
            true_label_seq.append(id_to_label[l])
            true_pred_seq.append(id_to_label[p])
    true_labels.append(true_label_seq)
    true_preds.append(true_pred_seq)

print(classification_report(true_labels, true_preds))
print("Overall F1:", f1_score(true_labels, true_preds))

              precision    recall  f1-score   support

         LOC       0.80      0.85      0.82      4834
         ORG       0.68      0.72      0.70      4677
         PER       0.85      0.89      0.87      4635

   micro avg       0.78      0.82      0.80     14146
   macro avg       0.78      0.82      0.80     14146
weighted avg       0.78      0.82      0.80     14146

Overall F1: 0.7963427628639191


In [10]:
# Draft solution scaffold (students complete TODO sections)
from datasets import load_dataset
import csv
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForCausalLM

# -----------------------------
# 6) Configuration
# -----------------------------
BERT_MODEL_ID = 'lab4/models_cache/wikiann_bert_ner/checkpoint-1000' # Chose whether to use the model you trained, or to load an already trained model like 'dslim/bert-base-NER'
LLAMA_MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'  # TODO: replace with the 1B Llama available to you
MAX_SAMPLES = 64
OUT_CSV = os.path.join(DATA_DIR, 'exercise_predictions.csv')

# -----------------------------
# 7) Load data
# -----------------------------
wikiann_eval = load_dataset('wikiann', 'en', split=f'validation[:{MAX_SAMPLES}]')
print('Loaded examples:', len(wikiann_eval))

# -----------------------------
# 8) Load models/tokenizers
# -----------------------------
# TODO: Load BERT tokenizer/model for token classification
bert_tok = AutoTokenizer.from_pretrained(BERT_MODEL_ID)
bert_model = AutoModelForTokenClassification.from_pretrained(BERT_MODEL_ID)
bert_model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
bert_model = bert_model.to(device)

# TODO: Load Llama tokenizer/model for generation/chat
llama_tok = AutoTokenizer.from_pretrained(LLAMA_MODEL_ID)
llama_model = AutoModelForCausalLM.from_pretrained(
    LLAMA_MODEL_ID,
    dtype=torch.float16,
    device_map='auto',
)
llama_model.eval()

# -----------------------------
# 9) Prediction helpers
# -----------------------------
def predict_bert_ner(text):
    """Return a compact string summary of BERT entities for one text."""
    # TODO: implement with the BERT token classification model.
    # Suggested output format: "PERSON: Barack Obama | ORG: Stanford University | LOC: California"
    inputs = bert_tok(text, return_tensors='pt', truncation=True).to(device)
    with torch.no_grad():
        outputs = bert_model(**inputs)
    logits = outputs.logits[0]
    pred_ids = logits.argmax(dim=-1).tolist()
    tokens = bert_tok.convert_ids_to_tokens(inputs['input_ids'][0].tolist())

    entities = {}
    current_entity = []
    current_type = None

    for token, pred_id in zip(tokens, pred_ids):
        if token in ('[CLS]', '[SEP]', '[PAD]'):
            if current_entity and current_type:
                entities.setdefault(current_type, []).append(' '.join(current_entity))
                current_entity = []
                current_type = None
            continue

        label = bert_model.config.id2label[pred_id]

        if label.startswith('B-'):
            if current_entity and current_type:
                entities.setdefault(current_type, []).append(' '.join(current_entity))
            current_type = label[2:]
            current_entity = [token.replace('##', '')]
        elif label.startswith('I-') and current_type:
            if token.startswith('##'):
                current_entity[-1] += token[2:]
            else:
                current_entity.append(token)
        else:
            if current_entity and current_type:
                entities.setdefault(current_type, []).append(' '.join(current_entity))
            current_entity = []
            current_type = None

    if current_entity and current_type:
        entities.setdefault(current_type, []).append(' '.join(current_entity))

    # Map to readable names matching gold label format
    label_display = {'PER': 'PERSON', 'LOC': 'LOCATION', 'ORG': 'ORG'}
    parts = []
    for etype, enames in entities.items():
        display = label_display.get(etype, etype)
        parts.append(f"{display}: {', '.join(enames)}")
    return ' | '.join(parts) if parts else 'O'

def predict_llama_ner(text):
    """Return Llama JSON-like NER output as a string for one text."""
    # TODO: implement prompting/generation for PERSON/ORG/LOC extraction.
    # Reuse the style used earlier in this notebook for chat prompting.
    prompt = (
        "Extract ALL named entities from the text. Be thorough and do not miss any.\n\n"
        "Entity types:\n"
        "- PERSON: full or partial names of people\n"
        "- LOCATION: countries, cities, towns, regions, geographical features\n"
        "- ORG: companies, organizations, institutions, teams, agencies, governments, political parties\n\n"
        "Examples:\n"
        "Text: 'Apple was founded by Steve Jobs in California.'\n"
        "JSON: {\"PERSON\": [\"Steve Jobs\"], \"LOCATION\": [\"California\"], \"ORG\": [\"Apple\"]}\n\n"
        "Text: 'The United Nations met in New York to discuss the Russian invasion.'\n"
        "JSON: {\"PERSON\": [], \"LOCATION\": [\"New York\"], \"ORG\": [\"United Nations\"]}\n\n"
        "Text: 'Barack Obama studied at Harvard University before becoming president.'\n"
        "JSON: {\"PERSON\": [\"Barack Obama\"], \"LOCATION\": [], \"ORG\": [\"Harvard University\"]}\n\n"
        f"Text: '{text}'\n"
        "JSON:"
    )
    messages = [
        {"role": "system", "content": "You are a precise NER assistant. Output only valid JSON with keys PERSON, LOCATION, ORG. No explanation, no markdown."},
        {"role": "user", "content": prompt},
    ]
    input_ids = llama_tok.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors='pt',
        tokenize=True,
    )
    # Make sure it's a plain tensor, not BatchEncoding
    if not isinstance(input_ids, torch.Tensor):
        input_ids = input_ids['input_ids']
    input_ids = input_ids.to(llama_model.device)
    attention_mask = torch.ones_like(input_ids)

    with torch.no_grad():
        output_ids = llama_model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=200,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=llama_tok.eos_token_id,
        )
    generated = output_ids[0][input_ids.shape[-1]:]
    raw = llama_tok.decode(generated, skip_special_tokens=True).strip()

    import json, re
    try:
        json_match = re.search(r'\{.*\}', raw, re.DOTALL)
        if json_match:
            entity_dict = json.loads(json_match.group())
            parts = []
            for etype in ['PERSON', 'LOCATION', 'ORG']:
                enames = entity_dict.get(etype, [])
                if enames:
                    parts.append(f"{etype}: {', '.join(enames)}")
            return ' | '.join(parts) if parts else 'O'
    except Exception:
        pass
    return raw

# -----------------------------
# 10) Run predictions
# -----------------------------
rows = []
for i, sample in enumerate(wikiann_eval):
    text = sample['tokens']
    if isinstance(text, list):
        text = ' '.join(text)

    # TODO: Uncomment after implementing prediction functions.
    bert_pred = predict_bert_ner(text)
    llama_pred = predict_llama_ner(text)

    # Temporary placeholders to test CSV writing first.
    # bert_pred = '<TODO_BERT_PREDICTION>'
    # llama_pred = '<TODO_LLAMA_PREDICTION>'

    rows.append({
        'id': i,
        'bert_prediction': bert_pred,
        'llama_prediction': llama_pred,
    })

print('Prepared rows:', len(rows))
print(rows[0] if rows else 'No rows')

# -----------------------------
# 11) Evaluate predictions
# -----------------------------
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

def extract_gold_bio_tags(wikiann_sample, label_name_map={'PER': 'PERSON', 'LOC': 'LOCATION', 'ORG': 'ORG'}):
    """Convert WikiANN BIO tags to BIO label sequences."""
    # WikiANN tag names are fixed: O, B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC
    tag_names_wikiann = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']

    tokens = wikiann_sample['tokens']
    raw_ner_tags = wikiann_sample.get('ner_tags', [])

    bio_tags = []
    for tag_idx in raw_ner_tags:
        raw_label = tag_names_wikiann[tag_idx] if tag_idx < len(tag_names_wikiann) else 'O'
        if raw_label == 'O':
            bio_tags.append('O')
        else:
            prefix, base_label = raw_label.split('-', 1)
            mapped_label = f'{prefix}-{label_name_map.get(base_label, base_label)}'
            bio_tags.append(mapped_label)
    return bio_tags

def parse_prediction_string_to_bio(pred_string, tokens):
    """Parse prediction string (e.g., 'PERSON: name | ORG: org') to BIO tags."""
    # Simple fallback: if prediction is placeholder, return all 'O'
    if pred_string.startswith('<TODO_') and pred_string.endswith('>'):
        return ['O'] * len(tokens)
    # TODO: Parse actual prediction strings (students implement this)
    import json, re
    bio_tags = ['O'] * len(tokens)

    # Try parsing as JSON first (Llama output)
    try:
        # Extract JSON object from the string in case there's surrounding text
        json_match = re.search(r'\{.*\}', pred_string, re.DOTALL)
        if json_match:
            entity_dict = json.loads(json_match.group())
            type_map = {'PERSON': 'PERSON', 'LOCATION': 'LOCATION', 'ORG': 'ORG'}
            for etype, enames in entity_dict.items():
                etype_mapped = type_map.get(etype.upper(), etype.upper())
                if not isinstance(enames, list):
                    continue
                for ename in enames:
                    ename_tokens = ename.strip().split()
                    # Find the entity span in tokens
                    for start in range(len(tokens) - len(ename_tokens) + 1):
                        if [t.lower() for t in tokens[start:start+len(ename_tokens)]] == [t.lower() for t in ename_tokens]:
                            bio_tags[start] = f'B-{etype_mapped}'
                            for k in range(1, len(ename_tokens)):
                                bio_tags[start + k] = f'I-{etype_mapped}'
                            break
            return bio_tags
    except Exception:
        pass

    # Try parsing as pipe-separated string (BERT output): "PERSON: name | LOC: place"
    try:
        for part in pred_string.split('|'):
            part = part.strip()
            if ':' not in part:
                continue
            etype, enames_str = part.split(':', 1)
            etype = etype.strip().upper()
            for ename in enames_str.split(','):
                ename = ename.strip()
                ename_tokens = ename.split()
                if not ename_tokens:
                    continue
                for start in range(len(tokens) - len(ename_tokens) + 1):
                    if [t.lower() for t in tokens[start:start+len(ename_tokens)]] == [t.lower() for t in ename_tokens]:
                        bio_tags[start] = f'B-{etype}'
                        for k in range(1, len(ename_tokens)):
                            bio_tags[start + k] = f'I-{etype}'
                        break
    except Exception:
        pass

    return bio_tags

# Collect gold and predicted sequences
gold_sequences = []
bert_sequences = []
llama_sequences = []

for i, sample in enumerate(wikiann_eval):
    gold_bio = extract_gold_bio_tags(sample)
    gold_sequences.append(gold_bio)

    bert_sequences.append(parse_prediction_string_to_bio(rows[i]['bert_prediction'], sample['tokens']))
    llama_sequences.append(parse_prediction_string_to_bio(rows[i]['llama_prediction'], sample['tokens']))

# Compute metrics
bert_precision = precision_score(gold_sequences, bert_sequences, zero_division=0)
bert_recall = recall_score(gold_sequences, bert_sequences, zero_division=0)
bert_f1 = f1_score(gold_sequences, bert_sequences, zero_division=0)

llama_precision = precision_score(gold_sequences, llama_sequences, zero_division=0)
llama_recall = recall_score(gold_sequences, llama_sequences, zero_division=0)
llama_f1 = f1_score(gold_sequences, llama_sequences, zero_division=0)

print('\n=== BERT NER Metrics ===')
print(f'Precision: {bert_precision:.4f}')
print(f'Recall: {bert_recall:.4f}')
print(f'F1: {bert_f1:.4f}')
print(classification_report(gold_sequences, bert_sequences, zero_division=0))

print('\n=== Llama NER Metrics ===')
print(f'Precision: {llama_precision:.4f}')
print(f'Recall: {llama_recall:.4f}')
print(f'F1: {llama_f1:.4f}')
print(classification_report(gold_sequences, llama_sequences, zero_division=0))

# -----------------------------
# 6) Save CSV
# -----------------------------
os.makedirs(DATA_DIR, exist_ok=True)
with open(OUT_CSV, 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['id', 'bert_prediction', 'llama_prediction'])
    writer.writeheader()
    writer.writerows(rows)

print('Saved:', OUT_CSV)

Loaded examples: 64


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Prepared rows: 64
{'id': 0, 'bert_prediction': 'ORG: Sioux Falls Arena | LOCATION: Sioux Falls , South Dakota', 'llama_prediction': 'LOCATION: Sioux Falls, South Dakota | ORG: Sioux Falls Arena'}

=== BERT NER Metrics ===
Precision: 0.6552
Recall: 0.6951
F1: 0.6746
              precision    recall  f1-score   support

    LOCATION       0.44      0.71      0.54        24
         ORG       0.82      0.60      0.69        30
      PERSON       0.85      0.79      0.81        28

   micro avg       0.66      0.70      0.67        82
   macro avg       0.70      0.70      0.68        82
weighted avg       0.72      0.70      0.69        82


=== Llama NER Metrics ===
Precision: 0.5875
Recall: 0.5732
F1: 0.5802
              precision    recall  f1-score   support

    LOCATION       0.36      0.54      0.43        24
         ORG       0.81      0.43      0.57        30
      PERSON       0.75      0.75      0.75        28

   micro avg       0.59      0.57      0.58        82
   macro a

In [6]:
import os
os.makedirs(os.path.join('lab2', 'data'), exist_ok=True)

with open(OUT_CSV, 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['id', 'bert_prediction', 'llama_prediction'])
    writer.writeheader()
    writer.writerows(rows)

print('Saved:', OUT_CSV)

Saved: lab2/data/exercise_predictions.csv


# How to submit
Once you have reached f1 of 0.4 with the bert model, and 0.2 with the decoder, you can submit the results to the portal, following these instructions.
You can submit as many times as you want, we will evaluate your last submission:

1) generate the `.csv` file at `OUT_CSV` (must be called `exercise_predictions.csv`)
2) save the notebook as `.py` (any name is fine)
3) download both and put them in a new empty directory called `exercise_predictions`
4) zip the directory `exercise_predictions`
5) go to https://www.codabench.org/competitions/16004/, register to the task using your email @studenti.unipd.it
6) go to the submission page and create a new submission uploading `exercise_predictions.zip`